# Feature Engineering

Load the cleaned train, validation, and test splits, then derive reusable features with train-fitted parameters applied consistently to validation and test.

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

data_dir = Path("../data/processed")

train_df = pd.read_csv(data_dir / "train_cleaned.csv")
validation_df = pd.read_csv(data_dir / "validation_cleaned.csv")
test_df = pd.read_csv(data_dir / "test_cleaned.csv")

train_df.shape, validation_df.shape, test_df.shape

((315000, 38), (67500, 38), (67500, 38))

## Six Derived Features

1. `age_bucket`
- Computation: bin `age` into train-derived quantile buckets and encode the bucket labels.
- Captures: broad life-stage differences in click behavior without overfitting to exact ages.
- Train/apply: derive bin edges from `train_df` only, then reuse the same edges for validation and test.

2. `log_price_usd`
- Computation: apply `log1p(price_usd)`.
- Captures: compresses extreme price values so the model can learn smoother price-response patterns.
- Train/apply: deterministic transform with no fitting required.

3. `discounted_price`
- Computation: `price_usd * (1 - discount_pct)`.
- Captures: the effective paid price after discount, which can be more relevant than list price alone.
- Train/apply: deterministic transform with no fitting required.

4. `price_per_review`
- Computation: `price_usd / (review_count + 1)`.
- Captures: how expensive an item is relative to its social proof.
- Train/apply: deterministic transform with a small constant added to avoid division by zero.

5. `slot_position_zscore`
- Computation: standardize `slot_position` using the train mean and train standard deviation.
- Captures: relative ranking strength in a way that is comparable across the dataset.
- Train/apply: fit the mean and standard deviation on `train_df` only, then apply the same scaler to validation and test.

6. `frequency_density`
- Computation: average the existing frequency columns, `city_freq`, `product_name_freq`, and `brand_freq`.
- Captures: how common or familiar the impression is across the learned categorical frequency signals.
- Train/apply: no additional fitting is needed because the component frequency columns already come from the preprocessing step; the formula is applied identically to every split.


In [4]:
target_col = "clicked"
frequency_cols = ["city_freq", "product_name_freq", "brand_freq"]


def fit_feature_engineering(train_frame: pd.DataFrame) -> dict:
    age_values = train_frame["age"].dropna()
    age_edges = age_values.quantile([0.2, 0.4, 0.6, 0.8]).to_list()
    age_edges = [-np.inf, *sorted(set(age_edges)), np.inf]

    slot_mean = train_frame["slot_position"].mean()
    slot_std = train_frame["slot_position"].std(ddof=0)
    if pd.isna(slot_std) or slot_std == 0:
        slot_std = 1.0

    return {
        "age_edges": age_edges,
        "slot_mean": float(slot_mean),
        "slot_std": float(slot_std),
    }


def transform_feature_engineering(frame: pd.DataFrame, fitted: dict) -> pd.DataFrame:
    out = frame.copy()

    out["age_bucket"] = pd.cut(
        out["age"],
        bins=fitted["age_edges"],
        include_lowest=True,
        duplicates="drop",
    ).astype(str)

    out["log_price_usd"] = np.log1p(out["price_usd"])
    out["discounted_price"] = out["price_usd"] * (1 - out["discount_pct"])
    out["price_per_review"] = out["price_usd"] / (out["review_count"].fillna(0) + 1)
    out["slot_position_zscore"] = (out["slot_position"] - fitted["slot_mean"]) / fitted["slot_std"]
    out["frequency_density"] = out[frequency_cols].mean(axis=1)

    return out


fitted = fit_feature_engineering(train_df)
train_features = transform_feature_engineering(train_df, fitted)
validation_features = transform_feature_engineering(validation_df, fitted)
test_features = transform_feature_engineering(test_df, fitted)

train_features.shape, validation_features.shape, test_features.shape

((315000, 44), (67500, 44), (67500, 44))

In [5]:
output_dir = Path("../data/processed")

train_features.to_csv(output_dir / "train_feature_engineered.csv", index=False)
validation_features.to_csv(output_dir / "validation_feature_engineered.csv", index=False)
test_features.to_csv(output_dir / "test_feature_engineered.csv", index=False)

train_features.shape, validation_features.shape, test_features.shape

((315000, 44), (67500, 44), (67500, 44))

## Summary

- Six derived features were created for each split.
- The only train-fitted parameters were age bucket edges and the slot-position scaler.
- The transformed splits were saved to `data/processed/` as `train_feature_engineered.csv`, `validation_feature_engineered.csv`, and `test_feature_engineered.csv`.
